In [1]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint="https://ct-val.cognitiveservices.azure.com/",
    azure_deployment="gpt-4o-mini",
    api_version="2025-01-01-preview",
    api_key="1mrDubpfPwE2niMMIaKNRhNEX6o5jT9jOBXGl5rpPcEKhZzLyXzrJQQJ99CDACE1PydXJ3w3AAAAACOGNlJ4",
    temperature=0,
)

In [10]:
from typing import TypedDict, List
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    messages: List[HumanMessage]

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    print(f"\nAI: {response.content}")

graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()

user_input = ""
while user_input != "exit":
    user_input = input("Human: ")
    print(user_input)
    result = app.invoke({"messages": [HumanMessage(content=user_input)]})

exit

AI: It seems like you might want to end our conversation. If you have any more questions or need assistance in the future, feel free to reach out. Have a great day!


In [13]:
import os
from typing import TypedDict, List, Union
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI: {response.content}")
    return state

graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()

history = []
user_input = ""
while user_input != "exit":
    user_input = input("Human: ")
    history.append(HumanMessage(content=user_input))
    print(user_input)
    result = app.invoke({"messages": history})
    history = result["messages"]

Hi! My name is edgar

AI: Hi Edgar! How can I assist you today?
what is my name?

AI: Your name is Edgar. How can I help you today, Edgar?
List me all the messages we shared so far

AI: Sure! Here’s a summary of our conversation so far:

1. You introduced yourself: "Hi! My name is edgar."
2. I responded: "Hi Edgar! How can I assist you today?"
3. You asked: "What is my name?"
4. I replied: "Your name is Edgar. How can I help you today, Edgar?"
5. You requested: "List me all the messages we shared so far."

Let me know if there's anything else you'd like to discuss!
exit

AI: If you need to go, that's perfectly fine! Feel free to return anytime if you have more questions or need assistance. Have a great day, Edgar!


In [17]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def add(a: int, b: int) -> int:
    """Adds two numbers together"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together"""
    return a * b

tools = [add, multiply]
model = AzureChatOpenAI(
    azure_endpoint="https://ct-val.cognitiveservices.azure.com/",
    azure_deployment="gpt-4o-mini",
    api_version="2025-01-01-preview",
    api_key="1mrDubpfPwE2niMMIaKNRhNEX6o5jT9jOBXGl5rpPcEKhZzLyXzrJQQJ99CDACE1PydXJ3w3AAAAACOGNlJ4",
    temperature=0
).bind_tools(tools)

def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are my AI assistant, please answer my query to the best of your ability.")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

graph = StateGraph(AgentState)
graph.add_node("agent", model_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "end": END,
        "continue": "tools"
    }
)
graph.add_edge("tools", "agent")

app = graph.compile()

def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [("user", "Add 2+2 and then multiply the result by 6. Also tell me a joke, please")]}
print_stream(app.stream(inputs, stream_mode="values"))

================================ Human Message =================================

Add 2+2 and then multiply the result by 6. Also tell me a joke, please
================================== Ai Message ==================================
Tool Calls:
  add (call_D3hxh1W7uQdqeboab2j6PjK0)
 Call ID: call_D3hxh1W7uQdqeboab2j6PjK0
  Args:
    a: 2
    b: 2
  multiply (call_VWKIkSiaY33MrP5gIS24a0e4)
 Call ID: call_VWKIkSiaY33MrP5gIS24a0e4
  Args:
    a: 4
    b: 6
================================= Tool Message =================================
Name: multiply

24
================================== Ai Message ==================================

The result of adding 2 + 2 is 4, and when you multiply that by 6, you get 24.

And here's a joke for you: 
Why don't scientists trust atoms? 
Because they make up everything!


In [1]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_openai import AzureChatOpenAI

document_content = ""

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def update(content: str) -> str:
    """This tool updates the document with the content"""
    global document_content
    document_content = content
    return f"Document was updated successfully: {document_content}"

@tool
def save(filename: str) -> str:
    """This tool saves the document to the filename

    Args:
        filename: Name for the file to be saved
    """

    if not filename.endswith('.txt'):
        filename = f"{filename}.txt"

    try:
        with open(filename, "w") as f:
            f.write(document_content)
        print("Document was saved successfully to", filename)
        return f"Document was saved successfully to {filename}"
    except Exception as e:
        return f"Error saving document to {filename}"

tools = [update, save]

model = AzureChatOpenAI(
    azure_endpoint="https://ct-val.cognitiveservices.azure.com/",
    azure_deployment="gpt-4o-mini",
    api_version="2025-01-01-preview",
    api_key="1mrDubpfPwE2niMMIaKNRhNEX6o5jT9jOBXGl5rpPcEKhZzLyXzrJQQJ99CDACE1PydXJ3w3AAAAACOGNlJ4",
    temperature=0
).bind_tools(tools)

def agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content=f"""
    You are a Drafter, a helpful writing assistant. You are going to help the user update and modify documents.
    - If the user wants to update or modify content, user the 'update' tool with the complete updated content.
    - If the user wants to save and finish, you need to use the 'save' tool.
    - Make sure to always show the current document state after modifications.

    The current document content is: {document_content}
""")
    user_input = input("\nWhat would you like to do?")
    print(f"\nUser: {user_input}")
    user_message = HumanMessage(content=user_input)

    all_messages = [system_prompt] + list(state["messages"] + [user_message])

    response = model.invoke(all_messages)
    print(f"\nAI: {response.content}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

    return {"messages": [user_message, response]}

def should_continue(state: AgentState):
    messages = state["messages"]

    if not messages:
        return "continue"
    for message in reversed(messages):
        if (isinstance(message, ToolMessage) and
            "saved" in message.content.lower() and
            "document" in message.content.lower()):
            return "end"

    return "continue"

def print_stream(step):
        s = step
        if not s["messages"]:
            return
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        elif isinstance(message, ToolMessage):
            print("\nTOOL RESULT: ", message.content)
        else:
            message.pretty_print()

graph = StateGraph(AgentState)
graph.add_node("agent", agent)
tool_node = ToolNode(tools = tools)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_edge("agent", "tools")
graph.add_conditional_edges(
    "tools",
    should_continue,
    {
        "end": END,
        "continue": "agent"
    }
)

app = graph.compile()

def run_document_agent():
    print("=== Drafter ===")

    state = {"messages": []}

    for step in app.stream(state, stream_mode="values"):
        print_stream(step)

    print("=== Drafter Finished ===")

if __name__ == "__main__":
    run_document_agent()

=== Drafter ===

User: Hi!

AI: Hello! How can I assist you today?
================================== Ai Message ==================================

Hello! How can I assist you today?
================================== Ai Message ==================================

Hello! How can I assist you today?


KeyboardInterrupt: Interrupted by user

In [3]:
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage, HumanMessage, AIMessage
from operator import add as add_messages
from langchain_openai import AzureOpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool

model = AzureChatOpenAI(
    azure_endpoint="https://ct-val.cognitiveservices.azure.com/",
    azure_deployment="gpt-4o-mini",
    api_version="2025-01-01-preview",
    api_key="1mrDubpfPwE2niMMIaKNRhNEX6o5jT9jOBXGl5rpPcEKhZzLyXzrJQQJ99CDACE1PydXJ3w3AAAAACOGNlJ4",
    temperature=0
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-ada-002",
    azure_endpoint="https://ct-val.services.ai.azure.com",
    api_key="1mrDubpfPwE2niMMIaKNRhNEX6o5jT9jOBXGl5rpPcEKhZzLyXzrJQQJ99CDACE1PydXJ3w3AAAAACOGNlJ4"
)